# Fine-Tuning-Style Baseline And Limits

This notebook uses real provider calls with few-shot style examples, but no retrieved policy context. That makes the behavior-adaptation effect visible without running a real fine-tuning job.


## What This Proxy Can And Cannot Show

This notebook uses few-shot examples because they make behavior steering visible without running a training job. That is useful for strategy work: participants can see whether examples help tone, refusal style, or answer shape before deciding that real fine-tuning is worth the cost.

The proxy has hard limits. Few-shot examples are sent with one request and disappear afterward. They do not prove that a trained adapter would generalize, preserve safety behavior, or avoid regressions on unrelated tasks. Real fine-tuning still needs a held-out evaluation set and a rollback plan.


## Fine-Tuning Concepts Without Training Yet

**Supervised fine-tuning** teaches a model from examples of desired inputs and outputs. **Parameter-efficient fine-tuning** methods such as LoRA add or update a smaller set of parameters instead of changing every base-model weight. These approaches can improve repeated behavior, but they require careful evaluation because the model may also learn unwanted shortcuts.

In this strategy repo, we do not train. Instead, we use few-shot prompting as a low-cost probe. If examples do not help even in the prompt, real fine-tuning is unlikely to be the first thing to try. If examples help but the behavior is still inconsistent across held-out cases, that is evidence that a new fine-tuning run may be worth scoping with the workflow from the previous unit.


### Exercise

Do few-shot examples persist as model behavior after the request is finished?


### Solution

No. Few-shot examples condition only the current request; they do not update model weights or adapters.


## Visual Map: Behavior Adaptation Without Fresh Evidence

This notebook uses few-shot style examples as a live-call proxy for behavior adaptation. The visuals below connect back to what real parameter-efficient fine-tuning changes, while the code here deliberately does not train weights or adapters.

**LoRA adapter mechanism.** Source: [Hugging Face PEFT LoRA diagram](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/lora_diagram.png). This visual shows how an adapter can change model behavior without updating every base-model parameter.

<img src="assets/hf-lora-diagram.png" alt="LoRA adapter diagram" style="max-width: 760px; width: 100%; height: auto;">

---

The next visual shows a different parameter-efficient control mechanism, which helps avoid treating all adaptation methods as the same thing.

**Prefix tuning mechanism.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/peft/prefix-tuning.png). This visual gives another example of learned behavior control, which is different from retrieving fresh policy context at answer time.

<img src="assets/hf-prefix-tuning.png" alt="Prefix tuning diagram" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

Do LoRA or prefix tuning automatically provide fresh policy evidence at answer time?


### Solution

No. They can adapt model behavior, but fresh evidence still requires retrieval or another external context source.


In [ ]:
import json
from pprint import pprint

from pydantic import BaseModel, ValidationError

from strategy_helpers import (
    CONFIG,
    DOCUMENTS,
    REQUESTS,
    STYLE_EXAMPLES,
    ReplySource,
    call_model,
    compact_style_examples,
    style_messages,
)

## Load Shared Data And Runtime Helpers

All notebooks use the same fixtures and provider gateway so the comparisons stay controlled. The repeated setup lives in [`strategy_helpers.py`](strategy_helpers.py); lesson cells can now focus on the architecture decision being tested. Dry mode remains deterministic when no provider key is available.


In [ ]:
print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, "
    f"and {len(STYLE_EXAMPLES)} style examples."
)
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

## Inspect Few-Shot Style Messages


In [ ]:
# Exercise: list the unique tones represented in the examples.
# unique_tones = ...


In [ ]:
# Solution
unique_tones = sorted({example["tone"] for example in STYLE_EXAMPLES})
print(unique_tones)

In [ ]:
few_shot_messages = style_messages(STYLE_EXAMPLES)
print(f"Few-shot message count: {len(few_shot_messages)}")
print(compact_style_examples(STYLE_EXAMPLES))

In [ ]:
# Exercise: count how many chat messages the style examples create.
# few_shot_message_count = ...


In [ ]:
# Solution
few_shot_message_count = len(few_shot_messages)
print(few_shot_message_count)

## Call The Model Without Fresh Policy Context

The model receives style examples, but not policy documents. This intentionally tests the limitation: behavior can improve while factual grounding remains unproven.


## Reading Style-Only Outputs

Style-only outputs should be read with two questions in mind. First, did the answer follow the expected behavior pattern: concise, operational, careful refusal, or coaching? Second, did it claim policy facts that were not supplied in the prompt? A good style-only answer can still be incomplete because style examples do not carry source-of-truth policy evidence.

This is why the structured output includes `needs_policy_evidence`. That field is not a perfect judge, but it makes the limitation visible. When the model says it still needs policy evidence, it is admitting that behavior conditioning alone is not enough for grounded support answers.


## Examples Of Style-Only Success And Failure

A style-only answer succeeds when the task is mostly about response behavior. Example: “Reply in a concise operational tone and ask for the missing blocker.” The model can follow that pattern from examples.

A style-only answer fails when it needs current evidence. Example: “Can this duplicate payment be refunded?” The model may imitate the refund-response style, but it still needs the current refund policy to avoid inventing eligibility rules. This distinction is the reason strategy reviews separate behavior quality from grounding quality.


The style-only schema separates tone imitation from evidence needs. This is the central limitation of the notebook.


In [ ]:
class StyleOnlyAnswer(BaseModel):
    """Structured output for style-conditioned answers without retrieval."""

    answer: str
    predicted_tone: str
    needs_policy_evidence: bool
    response_source: ReplySource = "simulated"

Build the style-only prompt from examples and ask for structured JSON. Notice that no policy context is supplied.


Build the style-only prompt in a helper so it is clear that no policy context is included.


In [ ]:
def style_only_messages(question: str) -> list[dict[str, str]]:
    """Create a few-shot style prompt without retrieved evidence."""
    return [
        {
            "role": "system",
            "content": "Use the style of the examples. Return JSON with answer, predicted_tone, and needs_policy_evidence.",
        },
        *few_shot_messages,
        {"role": "user", "content": question},
    ]

The dry output intentionally says policy evidence is still needed.


In [ ]:
dry_style_only = json.dumps(
    {
        "answer": "Ask for the invoice number and both payment references in a concise support tone, but verify the current refund policy before giving exact eligibility details.",
        "predicted_tone": "concise",
        "needs_policy_evidence": True,
    }
)

Validate the structured style-only response before using it in comparisons.


In [ ]:
def style_only_answer(question: str) -> StyleOnlyAnswer:
    """Ask the provider to answer from style examples without policy context."""
    reply = call_model(
        style_only_messages(question),
        dry_response=dry_style_only,
        response_format={"type": "json_object"},
    )

    try:
        result = StyleOnlyAnswer.model_validate_json(reply.text)
        result.response_source = reply.source
        return result
    except ValidationError:
        return StyleOnlyAnswer(
            answer=reply.text,
            predicted_tone="unknown",
            needs_policy_evidence=True,
            response_source=reply.source,
        )

Run a few requests to show that style conditioning can be consistent while still needing policy evidence.


In [ ]:
for request in REQUESTS[:3]:
    print()
    print(request["request_id"], request["question"])
    pprint(style_only_answer(request["question"]).model_dump())

### Exercise

Why does the style-only answer still say policy evidence is needed?


### Solution

The examples teach tone and response shape, but they do not guarantee the current refund, MFA, or rate-limit policy facts.


## Compare What This Can And Cannot Solve

A style-conditioned call can learn the response pattern, but it still cannot prove the current refund window, rate limit, or MFA rule unless those facts are present in the prompt or already reliably known.


In [ ]:
# Summarize tone and evidence needs across the held-out requests.
summary_rows = []
for request in REQUESTS:
    output = style_only_answer(request["question"])
    summary_rows.append(
        {
            "request_id": request["request_id"],
            "expected_tone": request["preferred_tone"],
            "predicted_tone": output.predicted_tone,
            "response_source": output.response_source,
            "needs_policy_evidence": output.needs_policy_evidence,
        }
    )
pprint(summary_rows)

In [ ]:
fine_tuning_boundary = {
    "what_we_test_here": "real model behavior under few-shot style conditioning",
    "what_we_do_not_test_here": "whether a trained LoRA or hosted fine-tuned model generalizes",
    "prior_unit": "Fine-Tuning Small Models Locally + Evaluation",
}
pprint(fine_tuning_boundary)

## Further Reading

- [Hugging Face PEFT](https://huggingface.co/docs/peft/index) for real adapter training methods.
- [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer) for supervised fine-tuning workflows.
